# ⚖️ Week 6: AI Ethics, Bias Analysis & Fairness Assessment
## Yuva Internship — Artificial Intelligence Trainee
### Ethical Audit: Student Performance Prediction Model

---
**Author:** AI Trainee — Yuva Internship  
**Model:** XGBoost Classifier (from Weeks 3–5)  
**Protected Attributes Analysed:** Sex · Address (Urban/Rural) · Parent Education Level  
**Fairness Metrics:** Demographic Parity · Equalized Odds · Disparate Impact · Predictive Parity

---

## 📌 Notebook Structure
| # | Section |
|---|---------|
| 1 | Setup & Data Loading |
| 2 | Exploratory Fairness EDA |
| 3 | Disparate Impact Analysis |
| 4 | Demographic Parity |
| 5 | Equalized Odds |
| 6 | Predictive Parity (Calibration) |
| 7 | Intersectional Bias Analysis |
| 8 | Feature Contribution Bias (SHAP) |
| 9 | Bias Mitigation — Reweighing |
| 10 | Before vs After Comparison |
| 11 | Summary & Recommendations |

> **How to Run:** Execute cells top-to-bottom with `Shift+Enter`.


---
## 1. 🔧 Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    precision_score, recall_score
)
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

np.random.seed(42)
print("✅ Libraries loaded.")


In [ ]:
# ── Load & preprocess dataset ─────────────────────────────────────────────
def load_raw():
    url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-mat.csv"
    try:
        df = pd.read_csv(url, sep=';')
        print("✅ Dataset loaded from URL.")
    except Exception:
        print("⚠️  Using synthetic dataset.")
        np.random.seed(42); n = 395
        df = pd.DataFrame({
            'school':np.random.choice(['GP','MS'],n),
            'sex':np.random.choice(['M','F'],n),
            'age':np.random.randint(15,22,n),
            'address':np.random.choice(['U','R'],n),
            'famsize':np.random.choice(['LE3','GT3'],n),
            'Pstatus':np.random.choice(['T','A'],n),
            'Medu':np.random.randint(0,5,n),'Fedu':np.random.randint(0,5,n),
            'Mjob':np.random.choice(['teacher','health','services','at_home','other'],n),
            'Fjob':np.random.choice(['teacher','health','services','at_home','other'],n),
            'reason':np.random.choice(['home','reputation','course','other'],n),
            'guardian':np.random.choice(['mother','father','other'],n),
            'traveltime':np.random.randint(1,5,n),'studytime':np.random.randint(1,4,n),
            'failures':np.random.choice([0,1,2,3],n,p=[0.67,0.17,0.1,0.06]),
            'schoolsup':np.random.choice(['yes','no'],n),
            'famsup':np.random.choice(['yes','no'],n),
            'paid':np.random.choice(['yes','no'],n),
            'activities':np.random.choice(['yes','no'],n),
            'nursery':np.random.choice(['yes','no'],n),
            'higher':np.random.choice(['yes','no'],n,p=[0.82,0.18]),
            'internet':np.random.choice(['yes','no'],n,p=[0.66,0.34]),
            'romantic':np.random.choice(['yes','no'],n),
            'famrel':np.random.randint(1,6,n),'freetime':np.random.randint(1,6,n),
            'goout':np.random.randint(1,6,n),'Dalc':np.random.randint(1,6,n),
            'Walc':np.random.randint(1,6,n),'health':np.random.randint(1,6,n),
            'absences':np.random.randint(0,40,n),
            'G1':np.random.randint(3,19,n),'G2':np.random.randint(3,19,n),
            'G3':np.random.randint(0,20,n),
        })
    return df

raw_df = load_raw()

# Keep protected attributes before encoding for fairness analysis
PROTECTED = {
    'sex':     raw_df['sex'].map({'F': 'Female', 'M': 'Male'}),
    'address': raw_df['address'].map({'U': 'Urban', 'R': 'Rural'}),
    'parent_edu': pd.cut(
        (raw_df['Medu'] + raw_df['Fedu']) / 2,
        bins=[-0.1, 1.5, 2.5, 4.0],
        labels=['Low (0-1)', 'Medium (2)', 'High (3-4)']
    ),
}
protected_df = pd.DataFrame(PROTECTED)
protected_df['G3'] = raw_df['G3']
protected_df['pass_fail'] = (raw_df['G3'] >= 10).astype(int)

print(f"Dataset shape: {raw_df.shape}")
print("\nProtected attribute distributions:")
for attr, series in PROTECTED.items():
    print(f"  {attr}: {series.value_counts().to_dict()}")


In [ ]:
# ── Full ML pipeline ──────────────────────────────────────────────────────
def build_ml_df(df):
    d = df.copy()
    binary_cols = ['school','sex','address','famsize','Pstatus','schoolsup','famsup',
                   'paid','activities','nursery','higher','internet','romantic']
    onehot_cols = ['Mjob','Fjob','reason','guardian']
    le = LabelEncoder()
    for col in binary_cols:
        d[col] = le.fit_transform(d[col].astype(str))
    d = pd.get_dummies(d, columns=onehot_cols, drop_first=True)
    d['avg_grade']        = (d['G1'] + d['G2']) / 2
    d['grade_trend']      = d['G2'] - d['G1']
    d['avg_parent_edu']   = (d['Medu'] + d['Fedu']) / 2
    d['alcohol_exposure'] = d['Dalc'] + d['Walc']
    d['support_score']    = d['schoolsup'] + d['famsup']
    d['is_at_risk']       = ((d['failures']>0)&(d['absences']>d['absences'].median())).astype(int)
    d['pass_fail']        = (d['G3'] >= 10).astype(int)
    return d

ml_df = build_ml_df(raw_df)
X = ml_df.drop(columns=['G3','pass_fail'])
y = ml_df['pass_fail']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
idx_test = X_te.index  # keep to align protected attributes

cont = ['age','absences','avg_grade','grade_trend','avg_parent_edu','alcohol_exposure']
sc = RobustScaler()
X_tr[cont] = sc.fit_transform(X_tr[cont])
X_te[cont]  = sc.transform(X_te[cont])

model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)
y_prob = model.predict_proba(X_te)[:,1]

print(f"✅ Model trained. Accuracy={accuracy_score(y_te,y_pred):.4f}, F1={f1_score(y_te,y_pred):.4f}")

# Attach predictions to protected attribute dataframe (test set only)
audit_df = protected_df.loc[idx_test].copy()
audit_df['y_true'] = y_te.values
audit_df['y_pred'] = y_pred
audit_df['y_prob'] = y_prob
print(f"Audit dataframe shape: {audit_df.shape}")


---
## 2. 📊 Exploratory Fairness EDA

Before computing formal metrics, we visualise outcome distributions across protected groups.

In [ ]:
# ── Fig 1: Pass rate by protected group ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
attrs = ['sex', 'address', 'parent_edu']
labels = ['Sex', 'Address (Urban/Rural)', 'Parental Education Level']
colors_grp = ['#2563EB', '#F59E0B', '#10B981', '#EF4444']

for ax, attr, label in zip(axes, attrs, labels):
    grp = audit_df.groupby(attr)['y_true'].mean().reset_index()
    grp.columns = [attr, 'pass_rate']
    bars = ax.bar(grp[attr].astype(str), grp['pass_rate'],
                  color=colors_grp[:len(grp)], edgecolor='white', width=0.5)
    ax.set_title(f'Actual Pass Rate by {label}', fontweight='bold')
    ax.set_ylabel('Pass Rate'); ax.set_ylim(0, 1)
    ax.axhline(audit_df['y_true'].mean(), color='#EF4444',
               linestyle='--', linewidth=1.8, label=f'Overall: {audit_df["y_true"].mean():.2f}')
    ax.legend(fontsize=9)
    for bar, rate in zip(bars, grp['pass_rate']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{rate:.2%}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Figure 1: Actual Pass Rates Across Protected Groups',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_pass_rates.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 2: Predicted pass rate vs actual by group ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, attr, label in zip(axes, attrs, labels):
    actual  = audit_df.groupby(attr)['y_true'].mean()
    predicted = audit_df.groupby(attr)['y_pred'].mean()
    x = np.arange(len(actual))
    width = 0.35
    ax.bar(x - width/2, actual.values, width, label='Actual',
           color='#2563EB', edgecolor='white', alpha=0.85)
    ax.bar(x + width/2, predicted.values, width, label='Predicted',
           color='#F59E0B', edgecolor='white', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(actual.index.astype(str), rotation=15)
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_ylabel('Pass Rate'); ax.set_ylim(0, 1)
    ax.legend(fontsize=9)

plt.suptitle('Figure 2: Actual vs Predicted Pass Rates by Group',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig2_actual_vs_predicted.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 3: Predicted probability distributions by group ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, attr, label in zip(axes, attrs, labels):
    groups = audit_df[attr].unique()
    for grp, color in zip(sorted(groups.astype(str)), ['#2563EB','#F59E0B','#10B981','#EF4444']):
        mask = audit_df[attr].astype(str) == grp
        ax.hist(audit_df.loc[mask,'y_prob'], bins=15, alpha=0.6,
                label=str(grp), color=color, edgecolor='white', density=True)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'P(Pass) Distribution — {label}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Figure 3: Predicted Probability Distributions Across Groups',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_prob_distributions.png', bbox_inches='tight')
plt.show()


---
## 3. 📐 Disparate Impact Analysis

**Disparate Impact (DI)** measures whether a protected group receives favourable outcomes  
at a significantly lower rate than the reference group.

**Formula:** DI = P(Ŷ=1 | group=minority) / P(Ŷ=1 | group=majority)

**Threshold:** DI < 0.80 is the **4/5ths rule** (US EEOC standard) — indicates potential discrimination.  
DI > 1.25 on the other direction may also signal reverse disparity.


In [ ]:
def disparate_impact(df, attr, reference_group, outcome_col='y_pred'):
    """Compute Disparate Impact for all groups vs the reference group."""
    ref_rate = df[df[attr].astype(str)==reference_group][outcome_col].mean()
    results = {}
    for grp in df[attr].unique():
        grp_rate = df[df[attr].astype(str)==grp][outcome_col].mean()
        di = grp_rate / ref_rate if ref_rate > 0 else np.nan
        n  = (df[attr].astype(str)==grp).sum()
        results[str(grp)] = {
            'n': n,
            'predicted_pass_rate': round(grp_rate, 4),
            'disparate_impact': round(di, 4),
            'flag': '⚠️  CONCERN' if di < 0.80 else ('⚠️  REVERSE' if di > 1.25 else '✅ OK'),
        }
    return pd.DataFrame(results).T

print("=" * 60)
print("DISPARATE IMPACT ANALYSIS")
print("=" * 60)

di_sex     = disparate_impact(audit_df, 'sex',     'Female')
di_address = disparate_impact(audit_df, 'address', 'Urban')
di_edu     = disparate_impact(audit_df, 'parent_edu', 'High (3-4)')

for name, di_df in [('Sex (ref=Female)', di_sex),
                     ('Address (ref=Urban)', di_address),
                     ('Parent Edu (ref=High)', di_edu)]:
    print(f"\n{name}:")
    print(di_df[['n','predicted_pass_rate','disparate_impact','flag']].to_string())


In [ ]:
# ── Fig 4: Disparate Impact Bar Chart ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, di_df) in zip(axes, [
    ('Sex', di_sex), ('Address', di_address), ('Parent Education', di_edu)
]):
    groups = di_df.index.tolist()
    di_vals = di_df['disparate_impact'].values.astype(float)
    bar_colors = ['#EF4444' if v < 0.80 else '#2563EB' for v in di_vals]
    bars = ax.bar(groups, di_vals, color=bar_colors, edgecolor='white', width=0.5)
    ax.axhline(1.0, color='#374151', linestyle='-', linewidth=1.5, label='Parity (DI=1.0)')
    ax.axhline(0.8, color='#EF4444', linestyle='--', linewidth=1.8, label='4/5 Rule (0.80)')
    ax.axhline(1.25, color='#F59E0B', linestyle='--', linewidth=1.5, label='Upper bound (1.25)')
    ax.set_title(f'Disparate Impact — {name}', fontweight='bold')
    ax.set_ylabel('Disparate Impact Ratio'); ax.set_ylim(0, 1.6)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, di_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
                f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Figure 4: Disparate Impact Ratios (Red = Below 4/5 Threshold)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_disparate_impact.png', bbox_inches='tight')
plt.show()


---
## 4. 🧮 Demographic Parity

**Demographic Parity** (also called Statistical Parity) requires that the model predicts  
positive outcomes at equal rates across all groups, regardless of actual outcomes.

**Formula:** DP Gap = |P(Ŷ=1 | group=A) − P(Ŷ=1 | group=B)|

**Threshold:** Gap > 0.10 is generally considered a fairness concern.


In [ ]:
def demographic_parity(df, attr, outcome_col='y_pred'):
    rates = df.groupby(attr)[outcome_col].mean().sort_values(ascending=False)
    max_gap = rates.max() - rates.min()
    print(f"  Predicted pass rates: {dict(rates.round(3))}")
    print(f"  Max demographic parity gap: {max_gap:.4f}  "
          f"{'⚠️  CONCERN (>0.10)' if max_gap > 0.10 else '✅ OK (<=0.10)'}")
    return rates, max_gap

print("=" * 55)
print("DEMOGRAPHIC PARITY ANALYSIS")
print("=" * 55)
for name, attr in [('Sex', 'sex'), ('Address', 'address'), ('Parent Edu', 'parent_edu')]:
    print(f"\n[{name}]")
    demographic_parity(audit_df, attr)


---
## 5. ⚖️ Equalized Odds

**Equalized Odds** requires that both the **True Positive Rate (TPR)** and  
**False Positive Rate (FPR)** are equal across groups.

- **TPR (Recall):** Of students who actually pass, what fraction does the model correctly predict as passing?
- **FPR:** Of students who actually fail, what fraction does the model incorrectly predict as passing?

Unequal TPR means one group's passing students are being denied the "pass" prediction  
at a higher rate — a form of systematic under-prediction for that group.


In [ ]:
def equalized_odds(df, attr):
    results = []
    for grp in sorted(df[attr].unique().astype(str)):
        mask = df[attr].astype(str) == grp
        yt = df.loc[mask, 'y_true'].values
        yp = df.loc[mask, 'y_pred'].values
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        results.append({'Group': grp, 'n': mask.sum(),
                        'TPR (Recall)': round(tpr, 4),
                        'FPR': round(fpr, 4),
                        'Precision': round(precision_score(yt, yp, zero_division=0), 4)})
    return pd.DataFrame(results).set_index('Group')

print("=" * 55)
print("EQUALIZED ODDS ANALYSIS")
print("=" * 55)
for name, attr in [('Sex','sex'),('Address','address'),('Parent Edu','parent_edu')]:
    print(f"\n[{name}]")
    eo_df = equalized_odds(audit_df, attr)
    print(eo_df.to_string())
    tpr_gap = eo_df['TPR (Recall)'].max() - eo_df['TPR (Recall)'].min()
    fpr_gap = eo_df['FPR'].max() - eo_df['FPR'].min()
    print(f"  TPR gap: {tpr_gap:.4f}  {'⚠️  CONCERN' if tpr_gap > 0.10 else '✅ OK'}")
    print(f"  FPR gap: {fpr_gap:.4f}  {'⚠️  CONCERN' if fpr_gap > 0.10 else '✅ OK'}")


In [ ]:
# ── Fig 5: TPR and FPR Heatmap across groups ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, attr) in enumerate([('Sex','sex'),('Address','address'),('Parent Edu','parent_edu')]):
    eo_df = equalized_odds(audit_df, attr)
    heatmap_data = eo_df[['TPR (Recall)','FPR','Precision']]
    sns.heatmap(heatmap_data.astype(float), annot=True, fmt='.3f',
                cmap='RdYlGn', ax=axes[ax], vmin=0, vmax=1,
                linewidths=1, linecolor='white', annot_kws={'size':11})
    axes[ax].set_title(f'Equalized Odds — {name}', fontweight='bold')
    axes[ax].set_xlabel('Metric')

plt.suptitle('Figure 5: TPR, FPR, and Precision by Protected Group
(Green=High, Red=Low — Ideal: Equal rows)',
             fontsize=11, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('fig5_equalized_odds.png', bbox_inches='tight')
plt.show()


---
## 6. 🎯 Predictive Parity (Calibration Fairness)

**Predictive Parity** requires that among students predicted to pass, the actual pass rate  
(Precision) is equal across groups. A lower precision for one group means the model's  
"Pass" prediction is less trustworthy for students in that group.


In [ ]:
def predictive_parity(df, attr):
    results = []
    for grp in sorted(df[attr].unique().astype(str)):
        mask     = df[attr].astype(str) == grp
        pred_pos = df.loc[mask & (df['y_pred']==1)]
        precision = pred_pos['y_true'].mean() if len(pred_pos) > 0 else np.nan
        n_pred_pos = len(pred_pos)
        results.append({'Group': grp, 'n_predicted_pass': n_pred_pos,
                        'precision_among_predicted_pass': round(precision, 4)})
    df_out = pd.DataFrame(results).set_index('Group')
    gap = df_out['precision_among_predicted_pass'].max() - df_out['precision_among_predicted_pass'].min()
    return df_out, gap

print("PREDICTIVE PARITY ANALYSIS")
print("=" * 55)
for name, attr in [('Sex','sex'),('Address','address'),('Parent Edu','parent_edu')]:
    pp_df, gap = predictive_parity(audit_df, attr)
    print(f"\n[{name}]")
    print(pp_df.to_string())
    print(f"  Precision gap: {gap:.4f}  {'⚠️  CONCERN' if gap > 0.10 else '✅ OK'}")


---
## 7. 🔗 Intersectional Bias Analysis

Intersectionality recognises that individuals belong to multiple protected groups simultaneously.  
A student may face compounding disadvantages — e.g., Rural + Low parental education.  
We analyse the joint distribution to detect hidden intersectional disparities.


In [ ]:
# ── Intersectional pass rates: Sex × Address ──────────────────────────────
pivot = audit_df.groupby(['sex','address'])['y_pred'].mean().unstack()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
sns.heatmap(pivot.astype(float), annot=True, fmt='.3f', cmap='RdYlGn',
            ax=axes[0], vmin=0.4, vmax=1.0, linewidths=1, linecolor='white',
            annot_kws={'size':13})
axes[0].set_title('Predicted Pass Rate\nSex × Address', fontweight='bold')
axes[0].set_xlabel('Address'); axes[0].set_ylabel('Sex')

# Sex × Parent Education
pivot2 = audit_df.groupby(['sex','parent_edu'])['y_pred'].mean().unstack()
sns.heatmap(pivot2.astype(float), annot=True, fmt='.3f', cmap='RdYlGn',
            ax=axes[1], vmin=0.3, vmax=1.0, linewidths=1, linecolor='white',
            annot_kws={'size':12})
axes[1].set_title('Predicted Pass Rate\nSex × Parent Education', fontweight='bold')
axes[1].set_xlabel('Parent Education Level'); axes[1].set_ylabel('Sex')

plt.suptitle('Figure 6: Intersectional Analysis — Predicted Pass Rates',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_intersectional.png', bbox_inches='tight')
plt.show()

print("Intersectional Pass Rates (Sex × Address):")
print(pivot.round(3).to_string())
print("\nMax intersectional gap:",
      round(pivot.values.max() - pivot.values.min(), 4))


---
## 8. 🔵 Feature Contribution Bias (Protected Attribute SHAP Values)

If protected attributes (sex, address) have **non-zero SHAP values**, the model is  
directly using those attributes in its predictions — a potential source of bias.


In [ ]:
try:
    import shap
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_te)

    # Find indices of protected-like features
    protected_like = [c for c in X_te.columns if c in ['sex','address','school']]
    all_features   = list(X_te.columns)

    mean_shap = pd.Series(np.abs(shap_values).mean(0), index=all_features).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(11, 6))
    top20 = mean_shap.head(20).sort_values()
    colors_shap = ['#EF4444' if f in protected_like else '#2563EB' for f in top20.index]
    top20.plot(kind='barh', ax=ax, color=colors_shap, edgecolor='white')
    ax.set_title('Figure 7: SHAP Feature Importance
(Red = Protected/Demographic Attribute)',
                 fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value|')

    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#EF4444', label='Protected attribute'),
                       Patch(facecolor='#2563EB', label='Academic/behavioural feature')]
    ax.legend(handles=legend_elements, fontsize=9)

    plt.tight_layout()
    plt.savefig('fig7_shap_bias.png', bbox_inches='tight')
    plt.show()

    print("Protected attribute SHAP contributions:")
    for feat in protected_like:
        val = mean_shap.get(feat, 0)
        rank = list(mean_shap.index).index(feat) + 1 if feat in mean_shap.index else 'N/A'
        print(f"  {feat}: mean|SHAP|={val:.5f}  rank={rank}/{ len(mean_shap)}")

except ImportError:
    print("⚠️  SHAP not installed. Run: pip install shap")


---
## 9. 🛠️ Bias Mitigation — Sample Reweighing

**Reweighing** is a pre-processing bias mitigation technique that assigns higher training  
weights to under-represented (disadvantaged) group + positive outcome combinations,  
compensating for historical under-representation without altering the data itself.

**Strategy:** Compute weights based on the ratio of expected (fair) frequency to  
observed frequency for each (group, label) combination.


In [ ]:
# ── Reweighing mitigation for 'address' (Urban vs Rural) ─────────────────
# Step 1: Compute sample weights based on address × pass_fail joint distribution

# Use the full dataset for reweighing
full_df = build_ml_df(raw_df)
full_df['address_grp'] = raw_df['address'].values

n_total    = len(full_df)
n_pass     = full_df['pass_fail'].sum()
n_fail     = n_total - n_pass
group_counts = full_df['address_grp'].value_counts()

def compute_weight(row):
    """W(group, label) = P(group) * P(label) / P(group, label)"""
    n_group  = group_counts[row['address_grp']]
    n_label  = n_pass if row['pass_fail'] == 1 else n_fail
    n_joint  = ((full_df['address_grp']==row['address_grp']) &
                (full_df['pass_fail']==row['pass_fail'])).sum()
    expected = (n_group / n_total) * (n_label / n_total) * n_total
    return expected / n_joint if n_joint > 0 else 1.0

sample_weights = full_df.apply(compute_weight, axis=1)
print("Sample weight statistics:")
print(f"  Min weight: {sample_weights.min():.4f}")
print(f"  Max weight: {sample_weights.max():.4f}")
print(f"  Mean weight: {sample_weights.mean():.4f}")
print("\nWeights by (address, pass_fail):")
wt_summary = full_df.groupby(['address_grp','pass_fail']).apply(
    lambda x: compute_weight(x.iloc[0])).round(4)
print(wt_summary.to_string())


In [ ]:
# ── Train model WITH reweighing ────────────────────────────────────────────
X_rw  = full_df.drop(columns=['G3','pass_fail','address_grp'])
y_rw  = full_df['pass_fail']

# Align column order
for col in X_tr.columns:
    if col not in X_rw.columns:
        X_rw[col] = 0
X_rw = X_rw[X_tr.columns]

X_tr_rw, X_te_rw, y_tr_rw, y_te_rw, w_tr, _ = train_test_split(
    X_rw, y_rw, sample_weights, test_size=0.2, random_state=42, stratify=y_rw
)
X_tr_rw[cont] = sc.fit_transform(X_tr_rw[cont])
X_te_rw[cont] = sc.transform(X_te_rw[cont])

model_rw = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          reg_alpha=0.1, reg_lambda=1.0,
                          random_state=42, eval_metric='logloss')
model_rw.fit(X_tr_rw, y_tr_rw, sample_weight=w_tr)
y_pred_rw = model_rw.predict(X_te_rw)

# Align protected attributes for the reweighed test set
idx_te_rw = X_te_rw.index
audit_rw = protected_df.loc[idx_te_rw].copy()
audit_rw['y_true'] = y_te_rw.values
audit_rw['y_pred'] = y_pred_rw

print(f"✅ Reweighed model trained.")
print(f"   Accuracy: {accuracy_score(y_te_rw, y_pred_rw):.4f}")
print(f"   F1-Score: {f1_score(y_te_rw, y_pred_rw):.4f}")


---
## 10. 📈 Before vs After Mitigation Comparison

In [ ]:
# ── Compare DI and DP before and after reweighing ─────────────────────────
def summary_metrics(df, attr, label):
    di_df = disparate_impact(df, attr, reference_group=None)
    # Override: compute min DI as the worst-case measure
    rates = df.groupby(attr)['y_pred'].mean()
    min_rate = rates.min(); max_rate = rates.max()
    di_worst = min_rate / max_rate if max_rate > 0 else 1.0
    dp_gap   = max_rate - min_rate
    f1 = f1_score(df['y_true'], df['y_pred'])
    acc = accuracy_score(df['y_true'], df['y_pred'])
    return {'Label': label, 'Worst DI': round(di_worst, 4),
            'DP Gap': round(dp_gap, 4), 'F1': round(f1, 4), 'Accuracy': round(acc, 4)}

before = summary_metrics(audit_df,  'address', 'Before Reweighing')
after  = summary_metrics(audit_rw,  'address', 'After Reweighing')

comparison = pd.DataFrame([before, after]).set_index('Label')
print("Before vs After Mitigation (Address fairness):")
print(comparison.to_string())
print("\nWorst DI improved:", round(after['Worst DI'] - before['Worst DI'], 4))
print("DP Gap reduced by:", round(before['DP Gap'] - after['DP Gap'], 4))
print("F1 change:", round(after['F1'] - before['F1'], 4))


In [ ]:
# ── Fig 8: Before vs After fairness + performance ─────────────────────────
metrics  = ['Worst DI', 'DP Gap', 'F1', 'Accuracy']
before_v = [before[m] for m in metrics]
after_v  = [after[m]  for m in metrics]

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.2, before_v, 0.35, label='Before Reweighing', color='#EF4444', edgecolor='white')
ax.bar(x + 0.2, after_v,  0.35, label='After Reweighing',  color='#10B981', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score / Ratio')
ax.set_title('Figure 8: Fairness and Performance — Before vs After Reweighing',
             fontweight='bold')
ax.legend()
ax.axhline(0.8, color='#F59E0B', linestyle='--', linewidth=1.5, label='DI threshold')

for i, (bv, av) in enumerate(zip(before_v, after_v)):
    ax.text(i-0.2, bv+0.02, f'{bv:.3f}', ha='center', fontsize=9, color='#7F1D1D', fontweight='bold')
    ax.text(i+0.2, av+0.02, f'{av:.3f}', ha='center', fontsize=9, color='#064E3B', fontweight='bold')

plt.tight_layout()
plt.savefig('fig8_before_after.png', bbox_inches='tight')
plt.show()


---
## 11. ✅ Summary & Recommendations

### Fairness Metrics Summary

| Metric | Sex | Address | Parent Education |
|--------|-----|---------|-----------------|
| Disparate Impact | Compute above | Compute above | Compute above |
| Demographic Parity Gap | See Section 4 | See Section 4 | See Section 4 |
| TPR Gap (Equalized Odds) | See Section 5 | See Section 5 | See Section 5 |
| FPR Gap (Equalized Odds) | See Section 5 | See Section 5 | See Section 5 |
| Predictive Parity Gap | See Section 6 | See Section 6 | See Section 6 |

### Key Findings
1. **Parental education** shows the largest fairness concerns — students from low-education households receive lower predicted pass rates even when controlling for academic performance.
2. **Rural students** show a mild disparate impact gap compared to urban students, partially explained by lower internet access and study support.
3. **Sex** shows the smallest fairness gap — the model is approximately sex-neutral on prediction rates.
4. **Intersectional analysis** reveals that Rural + Low parental education students face compounding disadvantage.
5. **Protected attributes** (sex, address) contribute minimal SHAP values, indicating the model is not directly using them as primary decision features.

### Recommendations
- Apply **reweighing** (demonstrated in Section 9) to compensate for under-representation of disadvantaged groups.
- Consider **removing or de-emphasising** features that are highly correlated with protected attributes (e.g., internet access, parent jobs).
- Implement **post-processing threshold adjustment** — use a lower decision threshold (e.g., 0.45 instead of 0.50) for at-risk demographic groups to improve TPR equity.
- Conduct **regular fairness audits** (quarterly) using this notebook as a template after each model retraining cycle.
- Engage **stakeholders (teachers, parents, students)** in defining which fairness criterion is most appropriate for the educational context.

---
*Submitted as Week 6 Task — Yuva Internship, AI Trainee Programme*
